# DistilBERT tuning on the shared 20% split (report question (c))

Run on Google Colab, 8 Nov 2026, NVIDIA L4 GPU. Upload `train.json` and `test.json` to the Colab Files panel first.

Changes one setting at a time from the baseline (lr 2e-5, 4 epochs, unweighted loss, seed 42): class-weighted loss, lr 3e-5, lr 5e-5, the decision threshold, and seeds 1 and 2 for lr 2e-5 and 5e-5. The final model is the seed-42 baseline.

Outputs, copied into the repo: `results/distilbert_tuning/` (tuning table and per-epoch history), `results/distilbert_val_predictions.csv`, `results/distilbert_test_probabilities.csv` and `results/submission.csv`.

In [ ]:
# Step 1 check: GPU and data files
import os, torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
for f in ['train.json', 'test.json']:
    print(f, 'found' if os.path.exists(f) else 'MISSING', os.path.getsize(f) if os.path.exists(f) else '')

In [ ]:
# Step 2a: shared setup - identical data, split and tokenisation to notebooks/04_distilbert_20split.ipynb
import json, os, random, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

DEVICE = torch.device('cuda')
MODEL_NAME, MAX_LEN, BATCH_SIZE = 'distilbert-base-uncased', 256, 16
OUT = Path('runs'); OUT.mkdir(exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

train_df = pd.read_json('train.json'); test_df = pd.read_json('test.json')
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['reviews'].fillna('').tolist(), train_df['sentiments'].tolist(),
    test_size=0.2, random_state=42, stratify=train_df['sentiments'].tolist())
assert len(train_texts) == 5920 and len(val_texts) == 1481
print('split OK: 5920 train / 1481 validation;', sum(1 for y in val_labels if y == 0), 'negative in validation')

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
class ReviewDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.labels = labels
        self.enc = tokenizer([str(t).strip() for t in texts], truncation=True, padding='max_length',
                             max_length=MAX_LEN, return_tensors='pt')
    def __len__(self): return self.enc['input_ids'].shape[0]
    def __getitem__(self, i):
        item = {'input_ids': self.enc['input_ids'][i], 'attention_mask': self.enc['attention_mask'][i]}
        if self.labels is not None: item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item
train_ds, val_ds, test_ds = ReviewDataset(train_texts, train_labels), ReviewDataset(val_texts, val_labels), ReviewDataset(test_df['reviews'].tolist())
print('tokenised. Reviews truncated at 256 tokens:',
      round(100 * np.mean([len(tokenizer(t)['input_ids']) > 256 for t in train_df['reviews']]), 1), '%')

In [ ]:
# Step 2b: one function for every experiment. Changes ONE setting at a time vs the baseline.
RESULTS = []   # one row per run -> the question (c) table

def metrics(truth, preds):
    p, r, f, _ = precision_recall_fscore_support(truth, preds, labels=[0, 1], zero_division=0)
    tn, fp, fn, tp = confusion_matrix(truth, preds, labels=[0, 1]).ravel()
    return dict(accuracy=accuracy_score(truth, preds), macro_f1=f1_score(truth, preds, average='macro'),
                neg_precision=p[0], neg_recall=r[0], neg_f1=f[0], pos_precision=p[1], pos_recall=r[1], pos_f1=f[1],
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def predict_probs(model, ds):
    model.eval(); probs = []
    with torch.no_grad():
        for b in DataLoader(ds, batch_size=64):
            logits = model(input_ids=b['input_ids'].to(DEVICE), attention_mask=b['attention_mask'].to(DEVICE)).logits
            probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy().tolist())
    return np.array(probs)

def run(name, lr=2e-5, epochs=4, class_weighted=False, seed=42, weight_decay=0.01, warmup=0.1):
    set_seed(seed)
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total = len(loader) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(warmup * total), total)
    if class_weighted:   # same idea as class_weight='balanced' in scikit-learn: n / (2 * class count)
        counts = np.bincount(train_labels)
        w = torch.tensor(len(train_labels) / (2 * counts), dtype=torch.float).to(DEVICE)
    loss_fn = torch.nn.CrossEntropyLoss(weight=w if class_weighted else None)
    torch.cuda.reset_peak_memory_stats()
    best, history, train_s = None, [], 0.0
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.perf_counter(); tot = 0.0
        for b in loader:
            opt.zero_grad()
            logits = model(input_ids=b['input_ids'].to(DEVICE), attention_mask=b['attention_mask'].to(DEVICE)).logits
            loss = loss_fn(logits, b['labels'].to(DEVICE)); loss.backward(); tot += loss.item()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        train_s += time.perf_counter() - t0
        vp = predict_probs(model, val_ds); m = metrics(val_labels, (vp >= 0.5).astype(int))
        history.append(dict(run=name, epoch=ep, train_loss=tot / len(loader), **m))
        print(f"  epoch {ep}: train loss {tot/len(loader):.4f} | acc {m['accuracy']:.4f} | macro F1 {m['macro_f1']:.4f} | neg F1 {m['neg_f1']:.4f}")
        if best is None or m['macro_f1'] > best['macro_f1']:   # keep best checkpoint by validation macro F1
            best = dict(epoch=ep, val_probs=vp, **m)
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    tp_ = predict_probs(model, test_ds)
    row = dict(run=name, lr=lr, epochs=epochs, class_weighted=class_weighted, seed=seed,
               best_epoch=best['epoch'], train_seconds=round(train_s, 1),
               peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
               **{k: v for k, v in best.items() if k not in ('epoch', 'val_probs')})
    RESULTS.append(row)
    pd.DataFrame(history).to_csv(OUT / f'{name}_history.csv', index=False)
    pd.DataFrame({'reviews': val_texts, 'true': val_labels, 'prob_positive': best['val_probs']}).to_csv(OUT / f'{name}_val.csv', index=False)
    pd.DataFrame({'reviews': test_df['reviews'], 'prob_positive': tp_}).to_csv(OUT / f'{name}_test.csv', index=False)
    pd.DataFrame(RESULTS).to_csv(OUT / 'tuning_results.csv', index=False)
    del model; torch.cuda.empty_cache()
    print(f"{name}: best epoch {row['best_epoch']} | acc {row['accuracy']:.4f} | macro F1 {row['macro_f1']:.4f} | "
          f"neg P/R/F1 {row['neg_precision']:.3f}/{row['neg_recall']:.3f}/{row['neg_f1']:.3f} | {row['train_seconds']}s")
    return row
print('ready')

In [ ]:
# Step 2: baseline = current report settings (lr 2e-5, 4 epochs, no class weights, seed 42)
run('baseline')

In [ ]:
# Step 3: one change at a time vs the baseline
run('class_weighted', class_weighted=True)
run('lr_3e-5', lr=3e-5)
run('lr_5e-5', lr=5e-5)
display(pd.DataFrame(RESULTS)[['run','lr','class_weighted','seed','best_epoch','accuracy','macro_f1','neg_precision','neg_recall','neg_f1','pos_f1','train_seconds']].round(4))

In [ ]:
# Step 3b: decision threshold (no retraining). Uses the saved validation probabilities.
# Caveat: the threshold is chosen and scored on the same validation set, so any gain is optimistic.
for name in ['baseline', 'lr_5e-5']:
    v = pd.read_csv(OUT / f'{name}_val.csv')
    rows = []
    for t in np.arange(0.30, 0.91, 0.05):
        m = metrics(v['true'], (v['prob_positive'] >= t).astype(int))
        rows.append(dict(threshold=round(t, 2), macro_f1=m['macro_f1'], neg_precision=m['neg_precision'], neg_recall=m['neg_recall'], neg_f1=m['neg_f1']))
    df = pd.DataFrame(rows).round(4)
    print(name); print(df.to_string(index=False)); print()

In [ ]:
# Step 4: repeat the two closest candidates with seeds 1 and 2 (seed 42 already done)
for seed in [1, 2]:
    run(f'baseline_seed{seed}', seed=seed)
    run(f'lr_5e-5_seed{seed}', lr=5e-5, seed=seed)
res = pd.DataFrame(RESULTS)
res['config'] = np.where(res['class_weighted'], 'class_weighted', 'lr=' + res['lr'].astype(str))
summary = res[~res['class_weighted']].groupby('lr')[['accuracy','macro_f1','neg_f1','neg_recall','neg_precision','best_epoch']].agg(['mean','std','count']).round(4)
print(summary.to_string())
res.to_csv(OUT / 'tuning_results.csv', index=False)

In [ ]:
# Step 5: final model = lr 2e-5, seed 42 (run 'baseline'). Write submission.csv and download everything.
import shutil
t = pd.read_csv(OUT / 'baseline_test.csv')
sub = pd.DataFrame({'reviews': test_df['reviews'], 'predictions': (t['prob_positive'] >= 0.5).astype(int)})
assert len(sub) == 1851 and (sub['reviews'] == test_df['reviews']).all()
sub.to_csv(OUT / 'submission.csv', index=False, lineterminator='\n')
print('submission.csv:', len(sub), 'rows |', sub['predictions'].value_counts().to_dict())
archive = shutil.make_archive('distilbert_tuning_results', 'zip', OUT)
from google.colab import files
files.download(archive)
print('downloading', archive)